In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRegressor
import catboost
print("usinf catboost version",catboost.__version__)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold,cross_val_score,GridSearchCV,train_test_split
from sklearn.metrics import mean_squared_error,make_scorer

usinf catboost version 1.2.7


In [3]:
train_df = pd.read_csv("train_podcast.csv",sep=",")
display(train_df.head(3))
print(train_df.shape)

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
0,0,Mystery Matters,Episode 98,NaN,True Crime,74.81,Thursday,Night,NaN,0.0,Positive,31.41998
1,1,Joke Junction,Episode 26,119.8,Comedy,66.95,Saturday,Afternoon,75.95,2.0,Negative,88.01241
2,2,Study Sessions,Episode 16,73.9,Education,69.97,Tuesday,Evening,8.97,0.0,Negative,44.92531


(750000, 12)


In [5]:
print(train_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           750000 non-null  int64  
 1   Podcast_Name                 750000 non-null  object 
 2   Episode_Title                750000 non-null  object 
 3   Episode_Length_minutes       662907 non-null  float64
 4   Genre                        750000 non-null  object 
 5   Host_Popularity_percentage   750000 non-null  float64
 6   Publication_Day              750000 non-null  object 
 7   Publication_Time             750000 non-null  object 
 8   Guest_Popularity_percentage  603970 non-null  float64
 9   Number_of_Ads                749999 non-null  float64
 10  Episode_Sentiment            750000 non-null  object 
 11  Listening_Time_minutes       750000 non-null  float64
dtypes: float64(5), int64(1), object(6)
memory usage: 68.7+ MB


In [7]:
display(train_df.isnull().sum().sort_values(ascending=False).head(5))

Guest_Popularity_percentage    146030
Episode_Length_minutes          87093
Number_of_Ads                       1
id                                  0
Podcast_Name                        0
dtype: int64

In [9]:
train_df["Guest_Popularity_percentage"].fillna(0,inplace=True)
train_df["Episode_Length_minutes"].fillna(0,inplace=True)
train_df["Number_of_Ads"].fillna(0.0,inplace=True)

In [11]:
print(train_df[["Number_of_Ads","Episode_Length_minutes","Guest_Popularity_percentage"]].isnull().sum().sort_values(ascending=False))

Number_of_Ads                  0
Episode_Length_minutes         0
Guest_Popularity_percentage    0
dtype: int64


In [13]:
display(train_df.describe())

,id,Episode_Length_minutes,Host_Popularity_percentage,Guest_Popularity_percentage,Number_of_Ads,Listening_Time_minutes
count,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000
mean,374999.500000,57.014190,59.859901,42.065664,1.348853,45.437406
std,216506.495284,37.253678,22.873098,32.858857,1.151131,27.138306
min,0.000000,0.000000,1.300000,0.000000,0.000000,0.000000
25%,187499.750000,26.400000,39.410000,7.850000,0.000000,23.178350
50%,374999.500000,56.610000,60.050000,42.200000,1.000000,43.379460
75%,562499.250000,90.310000,79.530000,71.040000,2.000000,64.811580
max,749999.000000,325.240000,119.460000,119.910000,103.910000,119.970000


In [15]:
train_df.info(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           750000 non-null  int64  
 1   Podcast_Name                 750000 non-null  object 
 2   Episode_Title                750000 non-null  object 
 3   Episode_Length_minutes       750000 non-null  float64
 4   Genre                        750000 non-null  object 
 5   Host_Popularity_percentage   750000 non-null  float64
 6   Publication_Day              750000 non-null  object 
 7   Publication_Time             750000 non-null  object 
 8   Guest_Popularity_percentage  750000 non-null  float64
 9   Number_of_Ads                750000 non-null  float64
 10  Episode_Sentiment            750000 non-null  object 
 11  Listening_Time_minutes       750000 non-null  float64
dtypes: float64(5), int64(1), object(6)
memory usage: 68.7+ MB


In [42]:
cat_cols = train_df.select_dtypes(include=["object"]).columns
X_num = train_df.select_dtypes(include=["number"]).drop(columns=["Listening_Time_minutes"],axis=1)
label = LabelEncoder()
encoded_cols = []
for col in cat_cols:
    enc_col = f"{col}_enc"
    train_df[enc_col] = label.fit_transform(train_df[col])
    encoded_cols.append(enc_col)


X_cat = train_df[encoded_cols].to_numpy()  
X = np.hstack([X_num, X_cat])
y = train_df["Listening_Time_minutes"]

In [19]:
mse_scorer = make_scorer(mean_squared_error, greater_is_better
                         =False)
skf = KFold(n_splits=15, shuffle=True, random_state=700)

In [46]:
neigh = KNeighborsRegressor(n_neighbors=7,leaf_size=500,p=1,n_jobs=15,weights='uniform')
scores_neigh = cross_val_score(neigh,X,y,cv=skf,scoring=mse_scorer)
mse_values_neigh = -scores_neigh
print(f"fold mse for KNNeighbors : {mse_values_neigh}")
print(f"average mse for KNNeighbors : {mse_values_neigh.mean():.4f}")
neigh.fit(X,y)

fold mse for KNNeighbors : [354.19848456 355.47923517 354.21867224 357.74328075 352.52794589
 353.44327479 350.19844733 355.77517482 355.76851793 353.90219874
 348.58021735 352.18374238 351.17999368 353.73426003 351.36790889]
average mse for KNNeighbors : 353.3534


KNeighborsRegressor(leaf_size=500, n_jobs=15, n_neighbors=7, p=1)

In [56]:
X_train,y_train,X_test,y_test = train_test_split(X,y,test_size=0.8,random_state=700)
y_pred_test= neigh.predict(X_test)

ValueError: Expected a 2-dimensional container but got <class 'pandas.core.series.Series'> instead. Pass a DataFrame containing a single row (i.e. single sample) or a single column (i.e. single feature) instead.

In [21]:
test_df = pd.read_csv("test_podcast.csv")
test_df.shape

(250000, 11)

In [23]:
print(test_df.isnull().sum().sort_values(ascending=False).head(5))
print(test_df.info())

Guest_Popularity_percentage    48832
Episode_Length_minutes         28736
id                                 0
Podcast_Name                       0
Episode_Title                      0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 11 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           250000 non-null  int64  
 1   Podcast_Name                 250000 non-null  object 
 2   Episode_Title                250000 non-null  object 
 3   Episode_Length_minutes       221264 non-null  float64
 4   Genre                        250000 non-null  object 
 5   Host_Popularity_percentage   250000 non-null  float64
 6   Publication_Day              250000 non-null  object 
 7   Publication_Time             250000 non-null  object 
 8   Guest_Popularity_percentage  201168 non-null  float64
 9   Number_of_Ads                250000 n

In [25]:
test_df["Guest_Popularity_percentage"].fillna(0,inplace=True)
test_df["Episode_Length_minutes"].fillna(0,inplace=True)

In [27]:
print(test_df.isnull().sum().sort_values(ascending=False).head(5))

id                        0
Podcast_Name              0
Episode_Title             0
Episode_Length_minutes    0
Genre                     0
dtype: int64


In [58]:
cat_cols = test_df.select_dtypes(include=["object"]).columns
X_num = test_df.select_dtypes(include=["number"])
label = LabelEncoder()
encoded_cols = []
for col in cat_cols:
    enc_col = f"{col}_enc"
    test_df[enc_col] = label.fit_transform(test_df[col])
    encoded_cols.append(enc_col)


X_cat = test_df[encoded_cols].to_numpy()  
X_test1 = np.hstack([X_num, X_cat])
# 3) Prédiction
y_pred = neigh.predict(X_test1)

In [35]:
print(X_test.shape)

(250000, 17)


In [44]:
X.shape

(750000, 17)

In [50]:
podcast_df = pd.DataFrame({"id":test_df["id"],"Listening_Time_minutes":y_pred})
display(podcast_df)
podcast_df.to_csv("submission_podcast.csv",index=False)

,id,Listening_Time_minutes
0,750000,47.448827
1,750001,25.681171
2,750002,51.612737
3,750003,56.892259
4,750004,55.215871
...,...,...
249995,999995,30.877381
249996,999996,48.328673
249997,999997,16.847607
249998,999998,71.564681
